# Import Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import pickle

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

In [ ]:
RANDOM_STATE = 42

# Data Processing

## 22. Statlog Heart Disease

In [ ]:
df_statlog: pd.DataFrame = pd.read_csv('data/raw/22-statlog.csv')
df_statlog.head()

In [ ]:
df_statlog.info()

In [ ]:
df_statlog.isnull().sum()

In [ ]:
def preprocess_statlog(df: pd.DataFrame) -> tuple:
    df_new = df.copy()
    
    # Step 1: Rename columns
    df_new.rename(columns={
        "chest-pain": "cp_type",
        "rest-bp": "sbp",
        "serum-chol": "chol",
        "fasting-blood-sugar": "fbs",
        "electrocardiographic": "ecg",
        'max-heart-rate': 'hr',
        "major-vessels": "mv",
        "heart-disease": "target"
    }, inplace=True)
    
    # Step 2: Cast to int
    for col in ["age", "sex", "cp_type", "sbp", "chol", "fbs", "ecg", "hr", "angina", "slope", "mv", "thal", "target"]:
        df_new[col] = df_new[col].astype(int)
    
    # Step 3: Map values
    df_new['thal'] = df_new['thal'].map({3: 0, 6: 1, 7: 2})
    df_new['target'] = df_new['target'].map({1: 0, 2: 1})
    
    # Step 4: Drop duplicates
    df_new.drop_duplicates(inplace=True)
    
    # Step 5: Scale continuous features to [0, 1]
    cont_cols = ['age', 'sbp', 'chol', 'hr', 'oldpeak']
    scaler = MinMaxScaler()
    df_new[cont_cols] = scaler.fit_transform(df_new[cont_cols])
    
    df_new.attrs['dataset_name'] = 'Statlog'
    
    return df_new, scaler

In [ ]:
df_preprocessed_statlog, scaler_statlog = preprocess_statlog(df_statlog)
df_preprocessed_statlog.head()

<b>Statlog Column Definition</b>
1. age (Numeric, scaled)      - Patient Age [0, 1]
2. sex (Categorical)          - Patient Sex
    - 0 - female
    - 1 - male
3. cp_type (Categorical)      - Chest Pain Type
    - 1 - typical angina
    - 2 - atypical angina
    - 3 - nonanginal
    - 4 - asymptomatic
4. sbp (Numeric, scaled)      - Systolic Blood Pressure [0, 1]
5. chol (Numeric, scaled)     - Cholesterol [0, 1]
6. fbs (Categorical)          - Fasting Blood Sugar > 120 mg/dL
    - 0 - false
    - 1 - true
7. ecg (Categorical)          - Resting Electrocardiographic Result
    - 0 - normal
    - 1 - ST-T wave abnormality
    - 2 - left ventricular hypertrophy
8. hr (Numeric, scaled)       - Heart Rate [0, 1]
9. angina (Categorical)       - Exercise Induced Angina
    - 0 - no
    - 1 - yes
10. oldpeak (Numeric, scaled) - ST Depression [0, 1]
11. slope (Categorical)       - Slope of peak exercise ST segment
    - 1 - upsloping
    - 2 - flat
    - 3 - downsloping
12. mv (Numeric)              - Number of Major Vessels
13. thal (Categorical)        - Defect type
    - 0 - normal
    - 1 - fixed defect
    - 2 - reversible defect
14. target (Categorical)      - Disease Status
    - 0 - No Disease
    - 1 - Have Disease

In [ ]:
df_preprocessed_statlog.info()

In [ ]:
statlog_categorical_cols = ["sex", "cp_type", "fbs", "ecg", "angina", "slope", "thal", "target"]
statlog_numerical_cols = df_preprocessed_statlog.columns.difference(statlog_categorical_cols)

In [ ]:
def plot_categorical_distributions(df: pd.DataFrame, categorical_cols: list):
    for col in categorical_cols:
        plt.figure(figsize=(3, 5)) 
        sns.countplot(x=col, data=df)
        plt.title(f'Distribution of {col} in {df.attrs.get("dataset_name", "Unknown")} Dataset')
        plt.tight_layout()
        plt.show()
        
def plot_numerical_heatmap(df: pd.DataFrame, numerical_cols: list):
    sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm')
    plt.title(f'Correlation Heatmap of Numerical Features in {df.attrs.get("dataset_name", "Unknown")} Dataset')
    plt.show()

In [ ]:
plot_categorical_distributions(df_preprocessed_statlog, statlog_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_preprocessed_statlog, statlog_numerical_cols)

In [ ]:
def set_preprocessed(df: pd.DataFrame, dataset_name: str, scaler: MinMaxScaler, scaler_name: str) -> pd.DataFrame:
    df.to_csv(f'data/preprocessed/{dataset_name}.csv', index=False)
    scaler_file = f'scaler/{scaler_name}.pkl'
    with open(scaler_file, 'wb') as f:
        pickle.dump(scaler, f)
    
def get_preprocessed(dataset_name: str, scaler_name: str) -> tuple:
    df = pd.read_csv(f'data/preprocessed/{dataset_name}.csv')
    with open(f'scaler/{scaler_name}.pkl', 'rb') as f:
        scaler = pickle.load(f)
    return df, scaler

In [ ]:
set_preprocessed(df_preprocessed_statlog, '22-statlog-preprocessed', scaler_statlog, '22-scaler_statlog')

## 23. Coronary Heart Disease

In [ ]:
df_chd = pd.read_csv('data/raw/23-chd.csv')
df_chd.head()

In [ ]:
df_chd.info()

In [ ]:
df_chd.isnull().sum()

In [ ]:
def preprocess_chd(df: pd.DataFrame) -> tuple:
    df_new = df.copy()
        
    # Step 1: Map values
    df_new['famhist'] = df_new['famhist'].map({'Present': 1, 'Absent': 0})

    # Step 2: Rename columns
    df_new.rename(columns={
        'obesity': 'bmi',
        'chd': 'target'
    }, inplace=True)
    
    # Step 3: Drop duplicates
    df_new.drop_duplicates(inplace=True)
    
    # Step 4: Scale continuous features to [0, 1]
    cont_cols = ['sbp', 'tobacco', 'ldl', 'adiposity', 'typea', 'bmi', 'alcohol', 'age']
    scaler = MinMaxScaler()
    df_new[cont_cols] = scaler.fit_transform(df_new[cont_cols])
    
    df_new.attrs['dataset_name'] = 'CHD'
    
    return df_new, scaler

In [ ]:
df_preprocessed_chd, scaler_chd = preprocess_chd(df_chd)
df_preprocessed_chd.head()

<b>Coronary Heart Disease (CHD) Column Definition</b>
1. sbp (Numeric, scaled)      - Systolic Blood Pressure [0, 1]
2. tobacco (Numeric, scaled)  - Yearly Tobacco Use (kg) [0, 1]
3. ldl (Numeric, scaled)      - Low Density Lipoprotein [0, 1]
4. adiposity (Numeric, scaled) - Adiposity Measure [0, 1]
5. famhist (Categorical)      - Family History of Coronary Heart Disease
    - 0 - Absent
    - 1 - Present
6. typea (Numeric, scaled)    - Type A Personality Score [0, 1]
7. bmi (Numeric, scaled)  - Body Mass Index [0, 1]
8. alcohol (Numeric, scaled)  - Yearly Alcohol Consumption (liters) [0, 1]
9. age (Numeric, scaled)      - Patient Age [0, 1]
10. target (Categorical)     - Disease Status
    - 0 - No Disease
    - 1 - Have Disease

In [ ]:
df_preprocessed_chd.info()

In [ ]:
chd_categorical_cols = ["famhist", "target"]
chd_numerical_cols = df_preprocessed_chd.columns.difference(chd_categorical_cols)

In [ ]:
plot_categorical_distributions(df_preprocessed_chd, chd_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_preprocessed_chd, chd_numerical_cols)

In [ ]:
set_preprocessed(df_preprocessed_chd, '23-chd-preprocessed', scaler_chd, '23-chd-scaler')

## 24. Framingham

In [ ]:
df_framingham = pd.read_csv('data/raw/24-framingham.csv')
df_framingham.head()

In [ ]:
df_framingham.info()

In [ ]:
df_framingham.isnull().sum()

In [ ]:
for col in ["cigsPerDay", "totChol", "BMI", "heartRate", "glucose"]:
    df_framingham[col].plot(kind='hist', bins=30)
    plt.title(f'Distribution of {col} in Framingham Dataset')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
def preprocess_framingham(df: pd.DataFrame) -> tuple:
    df_new: pd.DataFrame = df.copy()
    
    # Step 1: Rename columns
    df_new.rename(columns={
        "Sex": "sex",
        "age": "age",
        "education": "edu",
        "currentSmoker": "smoking_status",
        "cigsPerDay": "num_cigs_per_day",
        "BPMeds": "bp_status",
        "prevalentStroke": "stroke_status",
        "prevalentHyp": "hypertension_status",
        "diabetes": "diabetes_status",
        "totChol": "chol",
        "sysBP": "sbp",
        "diaBP": "dbp",
        "BMI": "bmi",
        "heartRate": "hr",
        "glucose": "glucose_level",
        "TenYearCHD": "target"
    }, inplace=True)
    
    # Step 2: Map categorical values
    df_new['sex'] = df_new['sex'].map({"male": 1, "female": 0})
    df_new['smoking_status'] = df_new['smoking_status'].map({"Yes": 1, "No": 0})
    df_new['diabetes_status'] = df_new['diabetes_status'].map({"Yes": 1, "No": 0})
    
    # Step 3: Handle missing values
    df_new['edu'] = df_new['edu'].fillna(df_new['edu'].mode()[0])
    df_new['num_cigs_per_day'] = df_new['num_cigs_per_day'].fillna(df_new['num_cigs_per_day'].median())
    df_new["bp_status"] = df_new["bp_status"].fillna(df_new["bp_status"].mode()[0])
    df_new["chol"] = df_new["chol"].fillna(df_new["chol"].mean())
    df_new["bmi"] = df_new["bmi"].fillna(df_new["bmi"].mean())
    df_new["hr"] = df_new["hr"].fillna(df_new["hr"].mean())
    df_new["glucose_level"] = df_new["glucose_level"].fillna(df_new["glucose_level"].mean())
    
    # Step 4: Convert types
    df_new['edu'] = df_new['edu'].astype(int)
    df_new['bp_status'] = df_new['bp_status'].astype(int)
    df_new['stroke_status'] = df_new['stroke_status'].astype(int)
    df_new['hypertension_status'] = df_new['hypertension_status'].astype(int)
    df_new['smoking_status'] = df_new['smoking_status'].astype(int)
    df_new["target"] = df_new["target"].astype(int)

    # Step 5: Drop duplicates
    df_new.drop_duplicates(inplace=True)
    
    # Step 6: Scale continuous features to [0, 1]
    cont_cols = ['age', 'num_cigs_per_day', 'chol', 'sbp', 'dbp', 'bmi', 'hr', 'glucose_level']
    scaler = MinMaxScaler()
    df_new[cont_cols] = scaler.fit_transform(df_new[cont_cols])
    
    df_new.attrs['dataset_name'] = 'Framingham'
    
    return df_new, scaler

In [ ]:
df_preprocessed_framingham, scaler_framingham = preprocess_framingham(df_framingham)
df_preprocessed_framingham.head()

<b>Framingham Column Definition</b>
1. sex (Categorical)              - Patient Sex
    - 0 - female
    - 1 - male
2. age (Numeric, scaled)          - Patient Age [0, 1]
3. edu (Categorical)              - Education Level
    - 1 - Some high school
    - 2 - High school or GED
    - 3 - Some college or vocational school
    - 4 - College
4. smoking_status (Categorical)   - Current Smoker Status
    - 0 - No
    - 1 - Yes
5. num_cigs_per_day (Numeric, scaled) - Cigarettes per Day [0, 1]
6. bp_status (Categorical)        - On Blood Pressure Medication
    - 0 - No
    - 1 - Yes
7. stroke_status (Categorical)    - Prevalent Stroke
    - 0 - No
    - 1 - Yes
8. hypertension_status (Categorical) - Prevalent Hypertension
    - 0 - No
    - 1 - Yes
9. diabetes_status (Categorical)  - Diabetes Status
    - 0 - No
    - 1 - Yes
10. chol (Numeric, scaled)        - Total Cholesterol [0, 1]
11. sbp (Numeric, scaled)         - Systolic Blood Pressure [0, 1]
12. dbp (Numeric, scaled)         - Diastolic Blood Pressure [0, 1]
13. bmi (Numeric, scaled)         - Body Mass Index [0, 1]
14. hr (Numeric, scaled)          - Heart Rate [0, 1]
15. glucose_level (Numeric, scaled) - Glucose Level [0, 1]
16. target (Categorical)          - Disease Status
    - 0 - No Disease
    - 1 - Have Disease

In [ ]:
df_preprocessed_framingham.info()

In [ ]:
df_preprocessed_framingham.isnull().sum()

In [ ]:
framingham_categorical_cols = ['sex', 'edu', 'smoking_status', 'bp_status', 'stroke_status', 'hypertension_status', 'diabetes_status', 'target']
framingham_numerical_cols = df_preprocessed_framingham.columns.difference(framingham_categorical_cols)

In [ ]:
plot_categorical_distributions(df_preprocessed_framingham, framingham_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_preprocessed_framingham, framingham_numerical_cols)

In [ ]:
set_preprocessed(df_preprocessed_framingham, '24-framingham-preprocessed', scaler_framingham, '24-framingham-scaler')

## 25. Heart Disease

In [ ]:
df_heart = pd.read_csv('data/raw/25-heart.csv')
df_heart.head()

In [ ]:
df_heart.info()

In [ ]:
def preprocess_heart(df: pd.DataFrame) -> tuple:
    df_new = df.copy()
    
    # Step 1: Rename columns
    df_new.rename(columns={
        "cp": "cp_type",
        "trestbps": "sbp",
        "restecg": "ecg",
        "thalach": "hr",
        "exang": "angina",
        "ca": "mv"
    }, inplace=True)
    
    # Step 2: Map values
    df_new['cp_type'] = df_new['cp_type'].map({0: 1, 1: 2, 2: 3, 3: 4})
    df_new['slope'] = df_new['slope'].map({0: 1, 1: 2, 2: 3})

    # Step 3: Drop duplicates
    df_new.drop_duplicates(inplace=True)

    # Step 4: Scale continuous features to [0, 1]
    cont_cols = ['age', 'sbp', 'chol', 'hr', 'oldpeak']
    scaler = MinMaxScaler()
    df_new[cont_cols] = scaler.fit_transform(df_new[cont_cols])
    
    df_new.attrs['dataset_name'] = 'Heart'
    
    return df_new, scaler

In [ ]:
df_preprocessed_heart, heart_scaler = preprocess_heart(df_heart)
df_preprocessed_heart.head()

<b>Heart Disease Column Definition</b>
1. age (Numeric)            - Patient Age
2. sex (Categorical)        - Patient Sex
    - 0 - female
    - 1 - male
3. cp_type (Categorical)    - Chest Pain Type
    - 1 - typical angina
    - 2 - atypical angina
    - 3 - nonanginal
    - 4 - asymptomatic
4. sbp (Numeric)            - Systolic Blood Pressure
5. chol (Numeric)           - Cholesterol
6. fbs (Categorical)        - Fasting Blood Sugar > 120 mg/dL
    - 0 - false
    - 1 - true
7. ecg (Categorical)        - Resting Electrocardiographic Result
    - 0 - normal
    - 1 - ST-T wave abnormality
    - 2 - left ventricular hypertrophy
8. hr (Numeric)             - Heart Rate
9. angina (Categorical)     - Exercise Induced Angina
    - 0 - no
    - 1 - yes
10. oldpeak (Numeric)       - ST Depression induced by exercise relative to the rest
11. slope (Categorical)     - Slope of peak exercise ST segment
    - 1 - upsloping
    - 2 - flat
    - 3 - downsloping
12. mv (Numeric)            - Number of Major Vessels
13. thal (Categorical)      - Defect type
    - 0 - normal
    - 1 - fixed defect
    - 2 - reversible defect
    - 3 - unknown
14. target (Categorical)    - Disease Status
    - 0 - No Disease
    - 1 - Have Disease

In [ ]:
df_preprocessed_heart.info()

In [ ]:
df_preprocessed_heart.isnull().sum()

In [ ]:
heart_categorical_cols = ["sex", "cp_type", "fbs", "ecg", "angina", "slope", "thal", "target"]
heart_numerical_cols = df_preprocessed_heart.columns.difference(heart_categorical_cols)

In [ ]:
plot_categorical_distributions(df_preprocessed_heart, heart_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_preprocessed_heart, heart_numerical_cols)

In [ ]:
set_preprocessed(df_preprocessed_heart, '25-heart-preprocessed', heart_scaler, '25-heart-scaler')

## 26. Stroke

In [ ]:
df_stroke = pd.read_csv('data/raw/26-stroke.csv')
df_stroke.head()

In [ ]:
df_stroke["smoking_status"].value_counts()

In [ ]:
df_stroke.info()

In [ ]:
df_stroke.isnull().sum()

In [ ]:
df_stroke["bmi"].plot(kind='hist', bins=30)

In [ ]:
def preprocess_stroke(df: pd.DataFrame) -> tuple:
    df_new: pd.DataFrame = df.copy()
    
    # Step 0: Drop 'id' column
    df_new.drop(columns=['id'], inplace=True)
    
    # Step 1: Rename columns
    df_new.rename(columns={
        "gender": "sex",
        "age": "age",
        "hypertension": "hypertension_status",
        "heart_disease": "heart_disease_status",
        "ever_married": "marital_status",
        "work_type": "work_type",
        "Residence_type": "residence_type",
        "avg_glucose_level": "glucose_level",
        "bmi": "bmi",
        "smoking_status": "smoking_status",
        "stroke": "target"
    }, inplace=True)
    
    # Step 2: Map categorical values
    df_new['sex'] = df_new['sex'].map({"Male": 1, "Female": 0})
    df_new["marital_status"] = df_new["marital_status"].map({"Yes": 1, "No": 0})
    df_new["work_type"] = df_new["work_type"].map({"Private": 0, "Self-employed": 1, "Govt_job": 2, "children": 3, "Never_worked": 4})
    df_new["residence_type"] = df_new["residence_type"].map({"Urban": 1, "Rural": 0})
    df_new['smoking_status'] = df_new['smoking_status'].map({"formerly smoked": 0, "never smoked": 1, "smokes": 2, "Unknown": 3})

    df_new.dropna(subset=['sex'], inplace=True) # Others gender
    
    # Step 3: Convert types
    df_new['sex'] = df_new['sex'].astype(int)
    
    # Step 3: Handle missing values
    df_new["bmi"] = df_new["bmi"].fillna(df_new["bmi"].mean())

    # Step 4: Drop duplicates
    df_new.drop_duplicates(inplace=True)
    
    # Step 5: Scale continuous features to [0, 1]
    cont_cols = ["age", "glucose_level", "bmi"]
    scaler = MinMaxScaler()
    df_new[cont_cols] = scaler.fit_transform(df_new[cont_cols])
    
    df_new.attrs['dataset_name'] = 'Stroke'
    
    return df_new, scaler

In [ ]:
df_preprocessed_stroke, scaler_stroke = preprocess_stroke(df_stroke)
df_preprocessed_stroke.head()

<b>Stroke Column Description</b>
Based on the `preprocess_stroke` function and the sample data, here's the full column description:

1. sex (Categorical) — Patient Sex
   - 0 = Female
   - 1 = Male
2. age (Numeric, scaled) — Patient Age [0, 1]
3. hypertension_status (Categorical) — Hypertension
   - 0 = No
   - 1 = Yes
4. heart_disease_status (Categorical) — Heart Disease
   - 0 = No
   - 1 = Yes
5. marital_status (Categorical) — Ever Married
   - 0 = No
   - 1 = Yes
6. work_type (Categorical) — Type of Work
   - 0 = Private
   - 1 = Self-employed
   - 2 = Government Job
   - 3 = Children
   - 4 = Never Worked
7. residence_type (Categorical) — Residence Type
   - 0 = Rural
   - 1 = Urban
8. glucose_level (Numeric, scaled) — Average Glucose Level [0, 1]
9. bmi (Numeric, scaled) — Body Mass Index [0, 1]
10. smoking_status (Categorical) — Smoking Status
    - 0 = Formerly Smoked
    - 1 = Never Smoked
    - 2 = Smokes
    - 3 = Unknown
11. target (Categorical) — Disease Status
    - 0 = No Stroke
    - 1 = Had Stroke

In [ ]:
df_preprocessed_stroke.info()

In [ ]:
stroke_categorical_cols = ["sex", "hypertension_status", "heart_disease_status", "marital_status", "work_type", "residence_type", "smoking_status", "target"]
stroke_numerical_cols = df_preprocessed_stroke.columns.difference(stroke_categorical_cols)

In [ ]:
plot_categorical_distributions(df_preprocessed_stroke, stroke_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_preprocessed_stroke, stroke_numerical_cols)

In [ ]:
set_preprocessed(df_preprocessed_stroke, '26-stroke-preprocessed', scaler_stroke, '26-stroke-scaler')

## Concatenation

### Load Processed Datasets

In [ ]:
df_preprocessed_statlog, scaler_statlog = get_preprocessed('22-statlog-preprocessed', '22-scaler_statlog')
df_preprocessed_chd, scaler_chd = get_preprocessed('23-chd-preprocessed', '23-chd-scaler')
df_preprocessed_framingham, scaler_framingham = get_preprocessed('24-framingham-preprocessed', '24-framingham-scaler')
df_preprocessed_heart, scaler_heart = get_preprocessed('25-heart-preprocessed', '25-heart-scaler')
df_preprocessed_stroke, scaler_stroke = get_preprocessed('26-stroke-preprocessed', '26-stroke-scaler')

### 22 & 25

In [ ]:
df_statlog_heart = pd.concat([df_preprocessed_statlog, df_preprocessed_heart], ignore_index=True)
df_statlog_heart.attrs['dataset_name'] = 'Statlog-Heart'

In [ ]:
df_statlog_heart.info()

In [ ]:
df_statlog_heart.drop_duplicates(inplace=True)

In [ ]:
df_statlog_heart.info()

In [ ]:
statlog_heart_categorical_cols = ["sex", "cp_type", "fbs", "ecg", "angina", "slope", "thal", "target"]
statlog_heart_numerical_cols = df_statlog_heart.columns.difference(statlog_heart_categorical_cols)

In [ ]:
plot_categorical_distributions(df_statlog_heart, statlog_heart_categorical_cols)

In [ ]:
plot_numerical_heatmap(df_statlog_heart, statlog_heart_numerical_cols)